# MAT Research Pipeline

Purpose: visualize the Milestone 11 Mass Casualty Assessment Tool research pipeline across scan zones, synthetic survivor vital signs, localization, tracking, START-style triage, and local-only alert summaries.

Run path: install research extras with `uv sync --extra research`, open this notebook, and choose Run All. The cells are deterministic and do not require radio hardware, captured packets, or a completed `ruview.mat` implementation.

Fixture / simulated source: inline NumPy fixtures create one rectangular rubble scan zone, four operational WiFi transceivers, a buried survivor trajectory, synthetic breathing and heartbeat trends, motion confidence, range measurements, and local alert records. Guarded imports probe `ruview.mat`; local fallback helpers keep the notebook smoke-testable while Milestone 11 code workers are still in progress.

In [ ]:
import math

import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    raise RuntimeError('Install the research extra with: uv sync --extra research') from exc

try:
    import ruview.mat as mat_api
except Exception as exc:
    mat_api = None
    MAT_IMPORT_ERROR = exc
else:
    MAT_IMPORT_ERROR = None

try:
    from ruview import mat as mat_namespace
except Exception as exc:
    mat_namespace = mat_api
    MAT_NAMESPACE_IMPORT_ERROR = exc
else:
    MAT_NAMESPACE_IMPORT_ERROR = None

plt.rcParams.update({
    'figure.figsize': (11, 5),
    'axes.grid': True,
    'grid.alpha': 0.25,
})

API_IMPORT_STATUS = {
    'ruview.mat': 'available' if mat_api is not None else 'fallback',
    'ruview.mat namespace': 'available' if mat_namespace is not None else 'fallback',
}
print(API_IMPORT_STATUS)


In [ ]:
def _try_mat(name, *args, default=None):
    fn = getattr(mat_api, name, None) if mat_api is not None else None
    if callable(fn):
        try:
            return fn(*args), 'api'
        except Exception:
            pass
    return default, 'fallback'


def zone_contains(zone, xy):
    x, y = xy
    return zone['min_x'] <= x <= zone['max_x'] and zone['min_y'] <= y <= zone['max_y']


def fallback_triangulate(sensors_xy, ranges_m, zone, grid_step=0.1):
    xs = np.arange(zone['min_x'], zone['max_x'] + grid_step, grid_step)
    ys = np.arange(zone['min_y'], zone['max_y'] + grid_step, grid_step)
    best_xy = np.array([np.nan, np.nan])
    best_rmse = np.inf
    for x in xs:
        for y in ys:
            candidate = np.array([x, y])
            predicted = np.linalg.norm(sensors_xy - candidate, axis=1)
            rmse = float(np.sqrt(np.mean((predicted - ranges_m) ** 2)))
            if rmse < best_rmse:
                best_xy = candidate
                best_rmse = rmse
    uncertainty = min(5.0, 0.75 + 2.0 * best_rmse)
    return best_xy, best_rmse, uncertainty


def localize_series(sensors_xy, ranges_by_time, zone):
    api_result, source = _try_mat('localize_series', sensors_xy, ranges_by_time, zone)
    if api_result is not None:
        return api_result, source

    estimates = []
    residuals = []
    uncertainties = []
    for ranges_m in ranges_by_time:
        xy, residual, uncertainty = fallback_triangulate(sensors_xy, ranges_m, zone)
        estimates.append(xy)
        residuals.append(residual)
        uncertainties.append(uncertainty)
    return {
        'xy': np.asarray(estimates),
        'residual_m': np.asarray(residuals),
        'uncertainty_m': np.asarray(uncertainties),
    }, source


def smooth_track(xy, alpha=0.35):
    tracked = np.zeros_like(xy, dtype=float)
    tracked[0] = xy[0]
    for idx in range(1, len(xy)):
        tracked[idx] = alpha * xy[idx] + (1.0 - alpha) * tracked[idx - 1]
    return tracked


def classify_triage(breathing_bpm, heart_bpm, movement_score):
    statuses = []
    priority = []
    for breathing, heart, movement in zip(breathing_bpm, heart_bpm, movement_score):
        if breathing <= 0.1 and heart <= 0.1:
            status = 'Deceased'
            score = 4
        elif breathing < 10.0 or breathing > 30.0 or movement < 0.2:
            status = 'Immediate'
            score = 1
        elif heart > 110.0 or movement < 0.5:
            status = 'Delayed'
            score = 2
        else:
            status = 'Minor'
            score = 3
        statuses.append(status)
        priority.append(score)
    return np.asarray(statuses, dtype=object), np.asarray(priority, dtype=int)


def local_alerts(statuses, priority, uncertainty_m, times_s):
    alerts = []
    for idx, (status, score, uncertainty) in enumerate(zip(statuses, priority, uncertainty_m)):
        if status in {'Immediate', 'Delayed'} and uncertainty <= 3.0:
            alerts.append({
                'time_s': float(times_s[idx]),
                'triage': str(status),
                'priority': int(score),
                'uncertainty_m': float(uncertainty),
                'handler': 'local_test_only',
            })
    return alerts


In [ ]:
zone = {
    'name': 'North rubble rectangle',
    'min_x': 0.0,
    'min_y': 0.0,
    'max_x': 12.0,
    'max_y': 8.0,
}
sensors_xy = np.array([
    [0.6, 0.7],
    [11.4, 0.8],
    [11.0, 7.2],
    [0.9, 7.4],
], dtype=float)
sensor_names = np.array(['AP-1', 'AP-2', 'AP-3', 'AP-4'])

times_s = np.arange(0, 180, 10, dtype=float)
phase = times_s / times_s[-1]
true_xy = np.column_stack([
    6.0 + 0.45 * np.sin(2.0 * np.pi * phase),
    4.2 + 0.25 * np.cos(1.5 * np.pi * phase),
])
depth_m = 1.8 + 0.15 * np.sin(np.pi * phase)

breathing_bpm = 18.0 + 16.0 / (1.0 + np.exp(-12.0 * (phase - 0.62)))
heart_bpm = 74.0 + 42.0 / (1.0 + np.exp(-10.0 * (phase - 0.58)))
movement_score = np.clip(0.78 - 0.52 * phase + 0.05 * np.sin(5.0 * np.pi * phase), 0.12, 0.9)
signal_quality = np.clip(0.82 - 0.18 * depth_m + 0.04 * np.cos(4.0 * np.pi * phase), 0.25, 0.9)

base_ranges = np.linalg.norm(sensors_xy[None, :, :] - true_xy[:, None, :], axis=2)
deterministic_bias = 0.08 * np.sin(times_s[:, None] / 20.0 + np.arange(len(sensors_xy))[None, :])
ranges_m = base_ranges + deterministic_bias + 0.05 * depth_m[:, None]

localization, localization_source = localize_series(sensors_xy, ranges_m, zone)
estimated_xy = localization['xy']
tracked_xy = smooth_track(estimated_xy)
localization_error_m = np.linalg.norm(tracked_xy - true_xy, axis=1)
triage_status, triage_priority = classify_triage(breathing_bpm, heart_bpm, movement_score)
alerts = local_alerts(triage_status, triage_priority, localization['uncertainty_m'], times_s)

summary = {
    'localization_source': localization_source,
    'inside_zone': bool(all(zone_contains(zone, xy) for xy in tracked_xy)),
    'max_tracking_error_m': round(float(np.max(localization_error_m)), 2),
    'final_triage': str(triage_status[-1]),
    'local_alert_count': len(alerts),
}
print(summary)


In [ ]:
priority_colors = {
    'Immediate': '#d62728',
    'Delayed': '#ffbf00',
    'Minor': '#2ca02c',
    'Deceased': '#111111',
    'Unknown': '#7f7f7f',
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

axes[0, 0].plot(true_xy[:, 0], true_xy[:, 1], color='#1f77b4', label='Synthetic survivor path')
axes[0, 0].plot(tracked_xy[:, 0], tracked_xy[:, 1], color='#d62728', linestyle='--', label='Tracked estimate')
axes[0, 0].scatter(sensors_xy[:, 0], sensors_xy[:, 1], marker='^', s=80, color='#2ca02c', label='WiFi transceivers')
for name, xy in zip(sensor_names, sensors_xy):
    axes[0, 0].annotate(name, xy, xytext=(4, 4), textcoords='offset points')
axes[0, 0].set_xlim(zone['min_x'] - 0.5, zone['max_x'] + 0.5)
axes[0, 0].set_ylim(zone['min_y'] - 0.5, zone['max_y'] + 0.5)
axes[0, 0].set_xlabel('Zone X position (m)')
axes[0, 0].set_ylabel('Zone Y position (m)')
axes[0, 0].set_title('Scan zone localization')
axes[0, 0].legend(loc='upper right')

axes[0, 1].plot(times_s, breathing_bpm, label='Breathing rate', color='#1f77b4')
axes[0, 1].plot(times_s, heart_bpm, label='Heart rate', color='#d62728')
axes[0, 1].axhspan(10, 30, color='#1f77b4', alpha=0.08, label='Breathing START watch band')
axes[0, 1].set_xlabel('Elapsed time (s)')
axes[0, 1].set_ylabel('Rate (bpm)')
axes[0, 1].set_title('Synthetic vital signs')
axes[0, 1].legend(loc='upper left')

axes[1, 0].plot(times_s, movement_score, color='#9467bd', label='Movement confidence')
axes[1, 0].plot(times_s, signal_quality, color='#17becf', label='Signal quality')
for status in sorted(set(triage_status)):
    mask = triage_status == status
    axes[1, 0].scatter(times_s[mask], np.full(np.sum(mask), 0.05), color=priority_colors[status], label=f'Triage: {status}', s=40)
axes[1, 0].set_xlabel('Elapsed time (s)')
axes[1, 0].set_ylabel('Score (0-1)')
axes[1, 0].set_title('Detection confidence and triage state')
axes[1, 0].set_ylim(0, 1.0)
axes[1, 0].legend(loc='upper right')

axes[1, 1].plot(times_s, localization_error_m, color='#8c564b', label='Tracking error')
axes[1, 1].plot(times_s, localization['uncertainty_m'], color='#7f7f7f', linestyle='--', label='Uncertainty estimate')
alert_times = [alert['time_s'] for alert in alerts]
if alert_times:
    axes[1, 1].scatter(alert_times, np.interp(alert_times, times_s, localization_error_m), marker='x', s=70, color='#d62728', label='Local test alert')
axes[1, 1].set_xlabel('Elapsed time (s)')
axes[1, 1].set_ylabel('Meters')
axes[1, 1].set_title('Localization error and local alerts')
axes[1, 1].legend(loc='upper left')

plt.show()


In [ ]:
print('Local/test-only alerts')
for alert in alerts[:6]:
    print(f"t={alert['time_s']:5.1f}s triage={alert['triage']:<9} priority={alert['priority']} uncertainty={alert['uncertainty_m']:.2f}m handler={alert['handler']}")
if len(alerts) > 6:
    print(f'... {len(alerts) - 6} additional local alerts suppressed in notebook display')


Expected interpretation: the tracked estimate should stay inside the rectangular scan zone and close to the synthetic survivor path, breathing should rise into the abnormal START watch band near the final third of the run, movement and signal quality should decline gradually, triage should move from Minor or Delayed toward Immediate as distress increases, and alerts should appear only as local/test-only records once the triage state is urgent and localization uncertainty is actionable.

Limitations: this is a deterministic visual fixture, not a validated RF propagation model, medical device, rescue dispatch system, or emergency notification integration. The local fallbacks approximate MAT concepts with readable NumPy helpers instead of Rust event sourcing, debris ML, calibrated CSI preprocessing, production-grade TDoA geometry, Kalman covariance, re-identification, or real responder workflows. Treat thresholds, triage labels, location error, and alert rows as notebook diagnostics only.